# 🧙 Day 04a: NumPy Tricks — Write Less, Do More

---

## 🎯 What You'll Master Today
- np.where, np.select for conditional logic
- np.einsum for elegant tensor operations
- np.vectorize, np.apply_along_axis
- Memory tricks: views, strides, structured arrays
- Speed patterns: replacing Python loops with vectorized NumPy

**Goal:** Optimize any Python loop into vectorized NumPy on a whiteboard.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  NumPy Tricks — Advanced Arsenal                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  CONDITIONAL:  np.where(cond, true_val, false_val)              ║
║                np.select([cond1, cond2], [val1, val2], default) ║
║  APPLY:        np.vectorize(func)  — NOT true vectorization    ║
║                np.apply_along_axis(func, axis, arr)             ║
║  TENSOR:       np.einsum('ij,jk->ik', A, B)  — Einstein sum   ║
║  MEMORY:       arr.strides, np.lib.stride_tricks                ║
║                np.shares_memory(a, b)                            ║
║  STRUCTURED:   np.dtype([('name', 'U10'), ('age', 'i4')])      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# np.where — Vectorized if-else
# ============================================================

arr = np.array([1, -2, 3, -4, 5])

# Replace negatives with 0
result = np.where(arr > 0, arr, 0)
print(f"Clip negatives: {result}")  # [1 0 3 0 5]

# Label: positive/negative
labels = np.where(arr > 0, 'pos', 'neg')
print(f"Labels: {labels}")

# Find indices where condition is True
indices = np.where(arr > 0)[0]
print(f"Positive indices: {indices}")  # [0, 2, 4]

# 2D example
matrix = np.array([[1, 2], [3, 4]])
print(f"\nWhere > 2: {np.where(matrix > 2, matrix, -1)}")

In [ ]:
# ============================================================
# np.select — Multiple conditions (chained if-elif-else)
# ============================================================

scores = np.array([92, 85, 67, 45, 78, 55, 98])

conditions = [
    scores >= 90,
    scores >= 80,
    scores >= 70,
    scores >= 60,
]
choices = ['A', 'B', 'C', 'D']

grades = np.select(conditions, choices, default='F')
print(f"Scores: {scores}")
print(f"Grades: {grades}")
# ['A' 'B' 'D' 'F' 'C' 'F' 'A']

print("\n💡 np.select = vectorized if-elif-elif-else chain")

In [ ]:
# ============================================================
# np.einsum — Einstein Summation (the swiss army knife)
# ============================================================

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
v = np.array([1, 2])

# Matrix multiply: C_ik = Σ_j A_ij * B_jk
C = np.einsum('ij,jk->ik', A, B)
print(f"Matmul: \n{C}")

# Dot product: Σ_i a_i * b_i
dot = np.einsum('i,i->', v, v)
print(f"\nDot product: {dot}")  # 5

# Outer product: C_ij = a_i * b_j
outer = np.einsum('i,j->ij', v, v)
print(f"\nOuter product: \n{outer}")

# Transpose: B_ji = A_ij
print(f"\nTranspose: \n{np.einsum('ij->ji', A)}")

# Trace: Σ_i A_ii
print(f"\nTrace: {np.einsum('ii->', A)}")  # 5

# Diagonal: A_ii
print(f"Diagonal: {np.einsum('ii->i', A)}")  # [1, 4]

# Element-wise multiply: C_ij = A_ij * B_ij
print(f"\nElement-wise: \n{np.einsum('ij,ij->ij', A, B)}")

# Batch matrix multiply: C_bij = Σ_k A_bik * B_bkj
batch_A = np.random.randn(3, 2, 4)  # 3 batches of 2x4
batch_B = np.random.randn(3, 4, 5)  # 3 batches of 4x5
batch_C = np.einsum('bij,bjk->bik', batch_A, batch_B)
print(f"\nBatch matmul: {batch_A.shape} @ {batch_B.shape} = {batch_C.shape}")

In [ ]:
# ============================================================
# np.vectorize and np.apply_along_axis
# ============================================================

# np.vectorize — wraps a scalar function to work on arrays
# ⚠️ NOT true vectorization — just a convenience wrapper!
def classify(x):
    if x > 0: return 'positive'
    elif x == 0: return 'zero'
    else: return 'negative'

vfunc = np.vectorize(classify)
arr = np.array([-2, 0, 3, -1, 5])
print(f"Vectorized classify: {vfunc(arr)}")

# np.apply_along_axis — apply function along an axis
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

# Sort each row
sorted_rows = np.apply_along_axis(sorted, 1, matrix[:, ::-1])
print(f"\nSorted rows: \n{sorted_rows}")

# Custom function on each column
def range_func(col):
    return col.max() - col.min()

col_ranges = np.apply_along_axis(range_func, 0, matrix)
print(f"Column ranges: {col_ranges}")

print("\n💡 Prefer broadcasting/where over vectorize — vectorize is slow!")

In [ ]:
# ============================================================
# Memory Tricks: Views, Copies, Strides
# ============================================================

# Views vs Copies — critical performance concept
original = np.arange(10)

# Slicing = VIEW (shares memory)
view = original[2:5]
print(f"Shares memory: {np.shares_memory(original, view)}")  # True
view[0] = 99
print(f"Original after view change: {original}")  # [0,1,99,3,4,...]

# Fancy indexing = COPY
original = np.arange(10)
copy = original[[2, 3, 4]]
print(f"\nShares memory: {np.shares_memory(original, copy)}")  # False
copy[0] = 99
print(f"Original after copy change: {original}")  # unchanged

# Strides — how NumPy navigates memory
arr = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int64)
print(f"\nShape: {arr.shape}")
print(f"Strides: {arr.strides}")  # (24, 8) — bytes to next row, next col
print(f"Item size: {arr.itemsize} bytes")

# Structured arrays — like mini-DataFrames
dt = np.dtype([('name', 'U10'), ('age', 'i4'), ('score', 'f8')])
people = np.array([('Alice', 30, 95.5), ('Bob', 25, 87.3)], dtype=dt)
print(f"\nStructured: {people}")
print(f"Names: {people['name']}")
print(f"High scorers: {people[people['score'] > 90]}")

In [ ]:
# ============================================================
# 🔥 Drill: Optimize Loops → Vectorized NumPy
# ============================================================
import time

n = 1_000_000
arr = np.random.randn(n)

# --- SLOW: Python loop ---
start = time.time()
result_loop = []
for x in arr:
    if x > 0:
        result_loop.append(x ** 2)
    else:
        result_loop.append(0)
result_loop = np.array(result_loop)
t_loop = time.time() - start

# --- FAST: Vectorized ---
start = time.time()
result_vec = np.where(arr > 0, arr ** 2, 0)
t_vec = time.time() - start

print(f"Loop:       {t_loop:.4f}s")
print(f"Vectorized: {t_vec:.4f}s")
print(f"Speedup:    {t_loop / t_vec:.0f}x")

# More optimization patterns:
x = np.random.randn(1000)

# Pattern: ReLU
relu = np.maximum(x, 0)

# Pattern: Clamp between [a, b]
clamped = np.clip(x, -1, 1)

# Pattern: Normalize to [0, 1]
normalized = (x - x.min()) / (x.max() - x.min())

# Pattern: Running sum / cumulative
cumsum = np.cumsum(x)

# Pattern: Differences (like derivative)
diffs = np.diff(x)

# Pattern: Unique values with counts
arr_int = np.random.randint(0, 5, 100)
vals, counts = np.unique(arr_int, return_counts=True)
print(f"\nUnique counts: {dict(zip(vals, counts))}")

---

## 🗺️ NumPy Tricks Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  NumPy Tricks — When to Use What                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  if-else on array  → np.where(cond, true, false)               ║
║  if-elif chain     → np.select([conds], [vals], default)       ║
║  tensor operations → np.einsum('ij,jk->ik', A, B)             ║
║  scalar func → arr → np.vectorize(func)  ⚠️ slow              ║
║  func per row/col  → np.apply_along_axis(func, axis, arr)     ║
║  clamp values      → np.clip(arr, low, high)                  ║
║  replace negatives → np.maximum(arr, 0)  (ReLU!)              ║
║  running total     → np.cumsum(arr)                             ║
║  differences       → np.diff(arr)                               ║
║  unique + counts   → np.unique(arr, return_counts=True)        ║
║                                                                   ║
║  MEMORY RULES:                                                   ║
║    slice = VIEW (shared)   fancy_index = COPY                   ║
║    reshape = VIEW          flatten = COPY (use ravel)           ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | np.where vs np.select? | where = single condition, select = multiple conditions (if-elif chain) |
| 2 | What is np.einsum? | Einstein summation — express any tensor op as subscript notation. Covers matmul, dot, outer, trace, batch ops |
| 3 | View vs copy in NumPy? | Slicing/reshape = view (shared memory). Fancy indexing/flatten = copy. Check with shares_memory() |
| 4 | Is np.vectorize fast? | No! It's just a convenience wrapper around Python loop. Use broadcasting/where instead |
| 5 | How to optimize Python loops with NumPy? | Replace with: where, clip, maximum, cumsum, diff, broadcasting. Typical 50-100x speedup |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • np.where = vectorized if-else                        │
## │  • np.select = vectorized if-elif chain                 │
## │  • np.einsum = one-stop tensor operations               │
## │  • Slice = view, fancy index = copy                     │
## │  • Always vectorize loops → 50-100x speedup            │
## │  • np.vectorize is NOT true vectorization               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 02 — Pandas: DataFrames & Series**